# Task 1 hyperparameter sweeps

One cell per model. `run_grid_search(task, model, parameter_grid)`:

- **a list** in `parameter_grid` = values to try (every combination of all lists is run)
- **a single value** (or a one-element list) = constant in every run
- **not mentioned** = the value from `submission/lib/config.py` (Task 1 section)

Every run trains on the 90 % training split and is scored on the 10 % validation split (validation MSE).
No refit on 100 %, nothing is written to `trained_models/` -> the submitted models stay untouched.

**Result:** `output/task1/<model>_<start time>.json` with the best combination and its full parameter set
(`best.parameters`, can be copied into `lib/config.py` as it is), the swept and constant values and the score of every run.
The file is rewritten after every run: interrupting a cell keeps the finished runs.

**Optional:** `rerun_finalists_on_other_splits=True` re-trains the best `n_finalists` (default 3) combinations on the
validation splits with seeds 1 and 2; the winner is then the best mean over the three splits (2 extra runs per finalist).
`show_training_log=True` prints the epoch lines of every run.

Run time per run on the RTX 5080 (Task 1 sweep, 7 Nov): SVD (k = 120) ~80 s, SVD++ (k = 500) ~6 min, bias model a few seconds.

In [ ]:
%load_ext autoreload
%autoreload 2

# grid_search.py lives in this folder and imports assignment1.py / lib/ from ../submission
import torch

from grid_search import print_sweep_overview, run_grid_search

print(torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

## Bias model (closed form)

In [ ]:
# a few seconds per run
bias_model_sweep = run_grid_search("task1", "bias_model", {
    "user_bias_regularization": [0.5, 1.0, 2.0, 3.0],
    "item_bias_regularization": [3.0, 5.0, 8.0],
})

## SVD

In [ ]:
# ~80 s per run (k = 120)
svd_sweep = run_grid_search("task1", "svd", {
    "user_bias_regularization": [1.0, 3.0],
    "item_bias_regularization": [3.0, 5.0],
    "n_factors": 120,
})

## SVD++

In [ ]:
# ~6 min per run (k = 500) -> 4 runs ~25 min; with the finalist check (2 finalists x 2 splits) ~50 min
svdpp_sweep = run_grid_search("task1", "svdpp", {
    "user_bias_regularization": [1.0, 3.0],
    "item_bias_regularization": [3.0, 5.0],
}, rerun_finalists_on_other_splits=False, n_finalists=2)

## Factorization machine

In [ ]:
factorization_machine_sweep = run_grid_search("task1", "factorization_machine", {
    "factor_regularization": [4.0, 8.0, 16.0],
    "n_activity_buckets": 10,
})

## Item-item kNN

In [ ]:
# CPU, a few minutes per run (neighbourhoods of all items)
item_knn_sweep = run_grid_search("task1", "item_knn", {
    "n_neighbors": [25, 50, 100],
    "similarity_shrinkage": 100.0,
})

## Overview

In [ ]:
# one line per sweep file in output/task1 (also works right after a kernel restart)
print_sweep_overview("task1")